# RAG System Prototype for Course Material Question Answering

This notebook demonstrates a small scale implementation of a Retrieval-Augmented Generation (RAG) system. The system processes and indexes sample course material, retrieves relevant chunks based on a user query, builds a prompt, and finally generating an answer using a language model (LLM).

In [1]:
# Uncomment and run the following cell to install required packages if needed
# !pip install sentence-transformers faiss-cpu openai
# !pip install transformers accelerate torch
# !pip install sacrebleu rouge-score bert-score
# !pip install nltk

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# LLM Agent Class

In [3]:
from openai import OpenAI

openAiKeyFile = open('openaikey.txt', 'r', encoding='utf-8')
OpenAI_API_KEY = openAiKeyFile.readline().strip()
openAiKeyFile.close()
openai_models = [
    "gpt-3.5-turbo",
    "gpt-3.5-turbo-16k",
    "gpt-4-turbo",
    "gpt-4",
    "gpt-4o-mini",
    "gpt-4o"
]
LLM_Model = openai_models[0]

# Define the LlmAgent class
class LlmAgent:
    def __init__(self, name, key, model, temperature):
        self.name = name
        self.temperature = temperature
        self.model = model
        self.client = OpenAI(api_key=key)
        self.messages = []

    def initializeAgent(self, system_prompt):
        self.messages.append({"role": "system", "content": system_prompt})

    def sendMessage(self, userinput):
        self.messages.append({"role": "user", "content": userinput})

    def getResponse(self):
        response = self.client.chat.completions.create(model=self.model, messages=self.messages)
        response_message = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": response_message})
        return response_message

# Define a function to create an agent
def create_agent(role_name, description, key, model=LLM_Model, temperature=1.0):
    agent = LlmAgent(name=role_name, key=key, model=model, temperature=temperature)
    system_prompt = f"""
    You are the {role_name}. {description}
    """
    agent.initializeAgent(system_prompt)
    return agent
    

# RAG Functions

In [4]:
import numpy as np
import nltk
from sentence_transformers import SentenceTransformer
import faiss

# download nltk data
nltk.download('all')

# take in text with punctuation and split into chunks
def chunk_text(text, chunk_size, overlap):
    sentences = nltk.sent_tokenize(text)
    chunks = []
    start = 0
    while start < len(sentences):
        chunk = sentences[start: start + chunk_size]
        chunks.append(' '.join(chunk))
        start += (chunk_size - overlap)
    return chunks

# def chunk_text(text, chunk_size, overlap):
#     """
#     Splits the text into chunks of a given size with an overlap to maintain context.
#     """
#     words = text.split()
#     chunks = []
#     start = 0
#     while start < len(words):
#         chunk = words[start: start + chunk_size]
#         chunks.append(' '.join(chunk))
#         start += (chunk_size - overlap)
#     return chunks

# Load a pre-trained Sentence Transformer model for embedding
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_texts(texts):
    """
    Computes vector embeddings for a list of texts using the pre-trained model.
    """
    embeddings = model.encode(texts)
    return embeddings

def build_faiss_index(embeddings):
    """
    Builds a FAISS index from the given embeddings for efficient similarity search.
    """
    dim = embeddings.shape[1]
    #index = faiss.IndexFlatL2(dim)
    index = faiss.IndexFlatIP(dim) # inner product
    faiss.normalize_L2(embeddings) # new =)
    index.add(embeddings.astype('float32'))
    #index.add(embeddings)
    return index

def keyword_score(query, chunk):
    query = query.split() # ["What", "is", "y"]
    chunk = chunk.split() # ["What", "is", "y"]

    query_words = {word.strip('.,!?').lower() for word in query}
    chunk_words = {word.strip('.,!?').lower() for word in chunk}
    return len(query_words.intersection(chunk_words))

def retrieve(query, chunks, index, k):
    """
    Retrieves the top-k most relevant text chunks for a given query.
    First, retieve 2*k chunks based on the embeddings.
    Second, sort them based on the actual keywords relevance.
    Third, picks the top-k chunks.
    """
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding) # new =)
    D, I = index.search(np.array(query_embedding).astype('float32'), 2 * k)
    retrieved_chunks = [chunks[i] for i in I[0]]

    retrieved_chunks.sort(key = lambda chunk: keyword_score(query, chunk), reverse = True)

    return retrieved_chunks[:k]
    

    #return retrieved_chunks
def build_prompt(query, retrieved_chunks):
    """
    Combines the user query with retrieved context chunks to build a prompt for the LLM.
    """
    context_chunks = "".join(retrieved_chunks)

    prompt = f"""You are a helpful teaching assistant answering questions. 
                Please read the **Context** and **nQuestion** below and try to extract a cohesive answer from the context.
                Do not use your own knowledge and stick to the context being given to you.
                Please give a brief answer that matches the context wording as much as possible.\n
                **Context**: {context_chunks}\n
                **Question**: {query}"""
    return prompt



[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /Users/754346/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /Users/754346/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /Users/754346/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /Users/754346/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /Users/754346/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nlt

# Processing and Indexing Course Material

In [5]:
import os
import numpy as np
from tqdm import tqdm

# 2.0: Helper to load all .txt files from given folders
def load_all_texts(dirs, extensions=(".txt",)):
    docs = {}
    for d in dirs:
        for fname in os.listdir(d):
            if fname.lower().endswith(extensions):
                path = os.path.join(d, fname)
                with open(path, "r", encoding="utf-8") as f:
                    docs[fname] = f.read()
    return docs

# 2.1: Load documents
directories = ["processed_transcripts", "lecture_notes"]
documents = load_all_texts(directories)

# 2.2: Chunk each document and collect all chunks

all_chunks = []
for fname, text in documents.items():
    print(f"\n--- Processing '{fname}' ---")
    
    # Chunk it
    chunks = chunk_text(text, chunk_size=10, overlap=3)
    
    # show a progress bar instead of printing "-" lines
    for _ in tqdm(chunks, desc=f"Chunking {fname}", unit="chunk"):
        pass
    
    all_chunks.extend(chunks)

print(f"\nTotal chunks created: {len(all_chunks)}")

# 2.3: Embed all chunks
embeddings = embed_texts(all_chunks)
embeddings_np = np.array(embeddings).astype("float32")

# 2.4: Build the FAISS index
index = build_faiss_index(embeddings_np)

print("✅ All documents processed, embedded, and indexed.")



--- Processing 'lecture-7.txt' ---


Chunking lecture-7.txt: 100%|██████████████████████████| 54/54 [00:00<00:00, 770382.37chunk/s]



--- Processing 'lecture-6.txt' ---


Chunking lecture-6.txt: 100%|█████████████████████████| 63/63 [00:00<00:00, 1164057.94chunk/s]



--- Processing 'lecture-4.txt' ---


...

Chunking Classification_Lecture_Rewritten.txt: 100%|█████| 2/2 [00:00<00:00, 101067.57chunk/s]



--- Processing 'Linear_Regression_Rewritten.txt' ---


Chunking Linear_Regression_Rewritten.txt: 100%|███████████| 2/2 [00:00<00:00, 80659.69chunk/s]


Total chunks created: 1344


✅ All documents processed, embedded, and indexed.


# Retrival and Prompting

In [7]:
# Step 3.1: Define a sample query
#query = "What programming paradigms does Python support?"
query = "Why does the SVD of a matrix reveal its rank?"

# Step 3.2: Retrieve the top-k relevant chunks from the course material
retrieved_chunks = retrieve(query, all_chunks, index, k=5)
print("\nRetrieved Chunks:\n")
for chunk in retrieved_chunks:
    print(chunk)
    print("-"*50)

# Step 3.3: Build a prompt combining the query and retrieved context
prompt = build_prompt(query, retrieved_chunks)
print("\nPrompt for LLM:\n")
print(prompt)
# display(Latex(prompt))


Retrieved Chunks:

again, a very beautiful characterization of the singular value. decomposition is the following: right, if Matrix a? I haven't spoken about rank far, but if Matrix a has rank R, okay, and R has to be less than equal to minimum of MN, and if m is greater than equal to n, it means that R is less than equal to n. R is the rank of the Matrix and that's exactly equal to the number of nonzero singular values. that's really the almost numerically the best way to determine the rank of a matrix is to do an SVD. okay, let me write it out. this means that Sigma 1 is greater than equal to Sigma 2, is greater than equal to Sigma r, and this is greater than zero, and what that means is that Sigma r + 1 = Sigma r + 2 to Sigma n, equal Z. okay, and then if I think about a, my Matrix a, then I have: a is r, n to r m. this is its Ma V1, as we saw to U1, Sigma 1, V2 to U2, Sigma 2, VR to ur, Sigma r and v? r + 1. it maps to zero because the rank is r, and similarly, V? r+ r, v, n is ma

In [8]:
# Step 3.4: Generate an answer using OpenAI LLM
the_llm_agent = create_agent(role_name="RAG Agent", description="", key=OpenAI_API_KEY, model=LLM_Model, temperature=0.0)
the_llm_agent.sendMessage(prompt)
answer = the_llm_agent.getResponse()
print("\nBig LLM Answer:\n")
print(answer)
#display(Latex(answer))


Big LLM Answer:

The SVD of a matrix reveals its rank because the number of nonzero singular values in the SVD is exactly equal to the rank of the matrix. The singular values in the SVD are ordered from greatest to smallest, and by examining these singular values, one can determine the rank of the matrix effectively.


# Evaluation Functions

In [9]:
# Step 4: Define the evaluation function

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import bert_score

# LLM evaluator
def evaluate_with_llm(reference_answer, generated_answer, model="gpt-4o", temperature=0.0):
    """
    Evaluates the quality of a generated answer against the reference answer using your custom LLM agent.
    """
    # Build the prompt
    prompt = f"""You are an expert evaluator.
You are given a reference answer and a generated answer.
Evaluate the generated answer based on how well it aligns with the reference answer.

Reference Answer:
{reference_answer}

Generated Answer:
{generated_answer}

Score the generated answer from 0 to 10:
- 0: Completely wrong, irrelevant, or nonsensical
- 5: Somewhat correct but incomplete, inaccurate, or partially aligned
- 10: Fully correct, well-aligned, and comprehensive

Only return the score as an integer number between 0 and 10. Nothing else."""
    
    try:
        # Create the evaluator agent
        evaluator_agent = create_agent(
            role_name="Evaluator Agent",
            description="An expert in evaluating alignment between answers.",
            key=OpenAI_API_KEY,
            model=model,
            temperature=temperature
        )
        
        # Send the prompt and get the response
        evaluator_agent.sendMessage(prompt)
        answer = evaluator_agent.getResponse()
        
        # Parse the score
        score_text = answer.strip()
        score = int(score_text)
        score = max(0, min(10, score))  # Clamp between 0 and 10
        
    except Exception as e:
        print(f"LLM evaluation error: {e}")
        score = 0
    
    return score

def evaluate(prediction, reference):
    """
    Evaluate a single prediction against a single reference using BLEU, ROUGE-L, BERTScore, and LLM_eval.
    Args:
        prediction (str): The generated answer.
        reference (str): The ground truth answer.
    Returns:
        dict: Dictionary with BLEU, ROUGE-L F1, BERTScore F1, and LLM evaluation score.
    """

    # Tokenize both
    pred_tokens = word_tokenize(prediction)
    ref_tokens = word_tokenize(reference)

    # BLEU (single sentence with smoothing)
    smoothing = SmoothingFunction().method1
    bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)

    # ROUGE-L (single sentence)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rougeL = scorer.score(reference, prediction)
    avg_rougeL = rougeL['rougeL'].fmeasure

    # BERTScore (single sentence) -- must wrap in list
    P, R, F1 = bert_score.score([prediction], [reference], lang="en", rescale_with_baseline=True)
    avg_bertF1 = float(F1.mean())

    # LLM Evaluation
    llm_score = evaluate_with_llm(reference, prediction)
    # print(f"Inside evaluation:\nReference: {reference}\nGenerated: {prediction}\nllm_score: {llm_score}\n")

    return {
        'bleu': bleu,
        'avg_rougeL': avg_rougeL,
        'avg_bertF1': max(0.0, avg_bertF1),
        'LLM_eval': llm_score
    }


# Evaluation Loop

In [10]:
import pandas as pd
import logging
from IPython.display import display, HTML, Markdown

logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# Read the CSV file
df = pd.read_csv("sample_questions/ML_questions.csv", encoding="utf-8")
   
# Normalize the column names
df.columns = (
    df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^\w_]", "", regex=True)
)

df_avgs = {}

for k in range(2,11):
    results = []
    # Loop through each model
    for model_name in openai_models:
        
        # RAG Loop
        for _, row in tqdm(df.iterrows(), 
                            total=len(df), 
                            desc=f"Evaluating {model_name}",):
            question  = row["question"]
            reference = row["answer"]
            
            chunks = retrieve(question, all_chunks, index, k=k)
            prompt = build_prompt(question, chunks)
            agent = create_agent(
                role_name="RAG Agent",
                description="You are a helpful TA.",
                key=OpenAI_API_KEY,
                model=model_name,
                temperature=0.0,
            )
            agent.sendMessage(prompt)
            prediction = agent.getResponse()
    
            # print(f"Question: {question}\nReference: {reference}\nPrediction: {prediction}\n")
            scores = evaluate(prediction, reference)
            scores["model"] = model_name
            results.append(scores)
    
    # Convert results to DataFrame and calculate average scores
    df_scores = pd.DataFrame(results)
    df_avg = (df_scores.groupby("model", as_index=False).mean())
    df_avgs[k] = df_avg
    
    # Now generate LaTeX table
    latex_table = "\\begin{table}[h]\n"
    latex_table += "\\centering\n"
    latex_table += "\\caption{Main results for k="+str(k)+". Higher is better.}\n"
    latex_table += "\\label{tab:main}\n"
    
    # Define the columns manually or from df_avg
    columns = df_avg.columns.tolist()
    
    # LaTeX header
    header = " & ".join(["\\textbf{" + col.replace("_", " ").title() + "}" for col in columns])
    latex_table += "\\begin{tabular}{" + "l" + "c" * (len(columns) - 1) + "}\n"
    latex_table += "\\toprule\n"
    latex_table += header + " \\\\\n"
    latex_table += "\\midrule\n"
    
    # Rows
    for _, row in df_avg.iterrows():
        row_data = " & ".join(
            f"{val:.1f}" if isinstance(val, (float, int)) else str(val) 
            for val in row
        )
        latex_table += row_data + " \\\\\n"
    
    latex_table += "\\bottomrule\n"
    latex_table += "\\end{tabular}\n"
    latex_table += "\\end{table}"
    
    # 1. Show an HTML preview of your averaged results
    html = df_avg.to_html(index=False,
                          float_format=lambda x: f"{x:.1f}")
    display(HTML(html))
    
    # 2. Then show the full LaTeX for copy-paste
    latex_block = f"""```latex
    {latex_table}
    ```"""
    display(Markdown(latex_block))


Evaluating gpt-4o: 100%|██████████████████████████████████████| 50/50 [02:05<00:00,  2.51s/it]


model,bleu,avg_rougeL,avg_bertF1,LLM_eval
gpt-3.5-turbo,0.0,0.2,0.3,7.6
gpt-3.5-turbo-16k,0.0,0.2,0.2,7.3
gpt-4,0.0,0.2,0.2,6.2
gpt-4-turbo,0.0,0.2,0.2,7.6
gpt-4o,0.0,0.2,0.2,7.4
gpt-4o-mini,0.0,0.2,0.2,7.5


```latex
    \begin{table}[h]
\centering
\caption{Main results for k=2. Higher is better.}
\label{tab:main}
\begin{tabular}{lcccc}
\toprule
\textbf{Model} & \textbf{Bleu} & \textbf{Avg Rougel} & \textbf{Avg Bertf1} & \textbf{Llm Eval} \\
\midrule
gpt-3.5-turbo & 0.0 & 0.2 & 0.3 & 7.6 \\
gpt-3.5-turbo-16k & 0.0 & 0.2 & 0.2 & 7.3 \\
gpt-4 & 0.0 & 0.2 & 0.2 & 6.2 \\
gpt-4-turbo & 0.0 & 0.2 & 0.2 & 7.6 \\
gpt-4o & 0.0 & 0.2 & 0.2 & 7.4 \\
gpt-4o-mini & 0.0 & 0.2 & 0.2 & 7.5 \\
\bottomrule
\end{tabular}
\end{table}
    ```

Evaluating gpt-4o: 100%|██████████████████████████████████████| 50/50 [02:16<00:00,  2.73s/it]


model,bleu,avg_rougeL,avg_bertF1,LLM_eval
gpt-3.5-turbo,0.0,0.2,0.3,8.0
gpt-3.5-turbo-16k,0.1,0.2,0.2,7.4
gpt-4,0.0,0.2,0.2,7.2
gpt-4-turbo,0.0,0.2,0.2,7.8
gpt-4o,0.0,0.2,0.2,7.7
gpt-4o-mini,0.0,0.2,0.2,7.6


...

model,bleu,avg_rougeL,avg_bertF1,LLM_eval
gpt-3.5-turbo,0.1,0.3,0.3,8.3
gpt-3.5-turbo-16k,0.1,0.3,0.3,8.4
gpt-4,0.0,0.2,0.2,7.8
gpt-4-turbo,0.0,0.2,0.2,8.3
gpt-4o,0.0,0.2,0.2,8.3
gpt-4o-mini,0.1,0.2,0.2,8.3


```latex
    \begin{table}[h]
\centering
\caption{Main results for k=9. Higher is better.}
\label{tab:main}
\begin{tabular}{lcccc}
\toprule
\textbf{Model} & \textbf{Bleu} & \textbf{Avg Rougel} & \textbf{Avg Bertf1} & \textbf{Llm Eval} \\
\midrule
gpt-3.5-turbo & 0.1 & 0.3 & 0.3 & 8.3 \\
gpt-3.5-turbo-16k & 0.1 & 0.3 & 0.3 & 8.4 \\
gpt-4 & 0.0 & 0.2 & 0.2 & 7.8 \\
gpt-4-turbo & 0.0 & 0.2 & 0.2 & 8.3 \\
gpt-4o & 0.0 & 0.2 & 0.2 & 8.3 \\
gpt-4o-mini & 0.1 & 0.2 & 0.2 & 8.3 \\
\bottomrule
\end{tabular}
\end{table}
    ```

Evaluating gpt-4o: 100%|██████████████████████████████████████| 50/50 [02:22<00:00,  2.85s/it]


model,bleu,avg_rougeL,avg_bertF1,LLM_eval
gpt-3.5-turbo,0.1,0.3,0.3,8.7
gpt-3.5-turbo-16k,0.1,0.2,0.3,8.1
gpt-4,0.0,0.2,0.2,8.0
gpt-4-turbo,0.0,0.2,0.2,8.7
gpt-4o,0.0,0.2,0.2,8.3
gpt-4o-mini,0.1,0.2,0.2,8.4


```latex
    \begin{table}[h]
\centering
\caption{Main results for k=10. Higher is better.}
\label{tab:main}
\begin{tabular}{lcccc}
\toprule
\textbf{Model} & \textbf{Bleu} & \textbf{Avg Rougel} & \textbf{Avg Bertf1} & \textbf{Llm Eval} \\
\midrule
gpt-3.5-turbo & 0.1 & 0.3 & 0.3 & 8.7 \\
gpt-3.5-turbo-16k & 0.1 & 0.2 & 0.3 & 8.1 \\
gpt-4 & 0.0 & 0.2 & 0.2 & 8.0 \\
gpt-4-turbo & 0.0 & 0.2 & 0.2 & 8.7 \\
gpt-4o & 0.0 & 0.2 & 0.2 & 8.3 \\
gpt-4o-mini & 0.1 & 0.2 & 0.2 & 8.4 \\
\bottomrule
\end{tabular}
\end{table}
    ```

# Show Evaluation Metrics

# Scratchpad

In [ ]:
# import os
# import re

# def clean_transcript(file_path):
#     # Derive the output filename by removing the trailing 'c' before .txt
#     base_name = os.path.basename(file_path)
#     if not base_name.endswith('c.txt'):
#         print(f"Skipped non-matching file: {file_path}")
#         return
    
#     output_name = base_name[:-5] + '.txt'  # remove 'c' and add '.txt'
#     output_path = os.path.join(os.path.dirname(file_path), output_name)

#     with open(file_path, 'r', encoding='utf-8') as infile:
#         lines = infile.readlines()

#     cleaned_lines = []
#     i = 0
#     while i < len(lines) - 2:
#         name1 = lines[i].strip()
#         name2 = lines[i + 1].strip()
#         time = lines[i + 2].strip()

#         if name1 == name2 and re.match(r'^\d{2}:\d{2}:\d{2}$', time):
#             i += 3  # skip these three lines
#         else:
#             cleaned_lines.append(lines[i])
#             i += 1

#     # Add any remaining lines at the end
#     while i < len(lines):
#         cleaned_lines.append(lines[i])
#         i += 1

#     with open(output_path, 'w', encoding='utf-8') as outfile:
#         outfile.writelines(cleaned_lines)

#     print(f"Cleaned file saved as: {output_path}")


# # Example usage
# clean_transcript("lecture-13c.txt")


In [ ]:
# from IPython.display import display, Latex

# # Display equation
# display(Math(r'E = mc^2'))

# # For text + equation
# display(Latex(r'The quadratic formula is: $x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$'))
